# 320 — Condition classification (audio / picture / reading)

Decodes the **stimulus condition** from a single electrode's spectro-temporal response.
3 classes, one sample per high-activity electrode × condition, run for **each feature
variant × each classifier** (logistic regression + random forest) = 6 experiments.

Every metric below comes from **nested GroupKFold by patient** — the outer fold holds out
whole patients for testing, the inner fold tunes hyper-parameters, and no patient ever
appears in both train and test. That makes the numbers an estimate of *generalisation to a
new patient*, not memorisation of these ones.

**How to read each run** (full narrative + figures in `390_results.ipynb`):
- **Balanced accuracy vs chance (0.333)** — headline separability, robust to imbalance.
- **Permutation p** — is balanced accuracy above a patient-shuffled-label null?
- **Confusion matrix** — which conditions get mixed up with which.
- **Per-class strength** — recall ± bootstrap CI, with FDR significance stars.
- **Feature importance** — which bands / time bins drove the separation.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_classify as C

# ---------------- config knobs ----------------
CLASSIFIERS  = ('logreg', 'rf')
OUTER_SPLITS = 5      # outer GroupKFold (held-out test patients)
INNER_SPLITS = 3      # inner GroupKFold (hyper-parameter tuning)
N_PERM       = 200    # label-permutation null reps (set 0 to skip; rf is the slow part)
N_BOOT       = 1000   # bootstrap reps for per-class CIs (cheap)
RANDOM_STATE = 42
print('classifiers:', CLASSIFIERS, '| outer/inner:', OUTER_SPLITS, INNER_SPLITS,
      '| n_perm:', N_PERM)


## Run all condition experiments
3 variants × 2 classifiers = 6 runs, each saved under
`outputs/classification/condition/<variant>/<classifier>/runs/<id>/`.


In [ ]:
manifests = []
for v in C.VARIANTS:
    X, y, groups, meta, cols = C.load_arrays('condition', None, v)
    for clf in CLASSIFIERS:
        m = C.run_experiment('condition', v, clf, X, y, groups, cols, meta,
                             outer_splits=OUTER_SPLITS, inner_splits=INNER_SPLITS,
                             n_perm=N_PERM, n_boot=N_BOOT, random_state=RANDOM_STATE)
        manifests.append(m)
print('\ndone:', len(manifests), 'runs')


## Summary table


In [ ]:
df = C.list_runs()
df = df[df.task == 'condition']
df[['variant', 'classifier', 'balanced_accuracy', 'chance_level',
    'macro_f1', 'permutation_p']].round(4)
